## Setup — Network and device names
Run this once. Every question uses these dictionaries.

In [ ]:
from collections import deque
network = {
    "N01": ["N02"],
    "N02": ["N01", "N03", "N04", "N30"],
    "N03": ["N02", "N04", "N05", "N06", "N07", "N12", "N13"],
    "N04": ["N02", "N03", "N08", "N09", "N10", "N11", "N14", "N15"],
    "N05": ["N03"],
    "N06": ["N03"],
    "N07": ["N03"],
    "N08": ["N04", "N09", "N18", "N23"],
    "N09": ["N04", "N08"],
    "N10": ["N04", "N11", "N17"],
    "N11": ["N04", "N10"],
    "N12": ["N03", "N16", "N17"],
    "N13": ["N03", "N18", "N19", "N20"],
    "N14": ["N04", "N21", "N22", "N23"],
    "N15": ["N04", "N24", "N25", "N26", "N27"],
    "N16": ["N12"],
    "N17": ["N10", "N12"],
    "N18": ["N08", "N13"],
    "N19": ["N13"],
    "N20": ["N13"],
    "N21": ["N14"],
    "N22": ["N14"],
    "N23": ["N08", "N14"],
    "N24": ["N15"],
    "N25": ["N15"],
    "N26": ["N15"],
    "N27": ["N15", "N28", "N29"],
    "N28": ["N27"],
    "N29": ["N27"],
    "N30": ["N02"]
}

device_names = {
    "N01": "Internet-Gateway",
    "N02": "Firewall-Main",
    "N03": "Core-Switch-1",
    "N04": "Core-Switch-2",
    "N05": "Web-Server",
    "N06": "Mail-Server",
    "N07": "DNS-Server",
    "N08": "Database-Server",
    "N09": "Auth-Server",
    "N10": "File-Server",
    "N11": "Backup-Server",
    "N12": "Switch-HR",
    "N13": "Switch-Finance",
    "N14": "Switch-Dev",
    "N15": "Switch-Lab",
    "N16": "HR-PC-1",
    "N17": "HR-PC-2",
    "N18": "Finance-PC-1",
    "N19": "Finance-PC-2",
    "N20": "Finance-Printer",
    "N21": "Dev-PC-1",
    "N22": "Dev-PC-2",
    "N23": "Build-Server",
    "N24": "Lab-PC-1",
    "N25": "Lab-PC-2",
    "N26": "Lab-PC-3",
    "N27": "WiFi-AP",
    "N28": "Guest-Laptop-1",
    "N29": "Guest-Laptop-2",
    "N30": "IDS-Sensor"
}

from collections import deque
print("Network loaded:", len(network), "devices")

## Question 1 — BFS order from N28
BFS uses a queue: first in, first out.

In [ ]:
def bfs(graph, start):
    queue = deque([start])
    visited = {start}
    order = []

    while queue:
        current = queue.popleft()
        order.append(current)
        for next_device in graph[current]:
            if next_device not in visited:
                visited.add(next_device)
                queue.append(next_device)
    return order

order = bfs(network, "N28")
print(" -> ".join(order))
print("Total:", len(order))

## Question 2 — BFS by hop
Each neighbor is one hop farther than its parent.

In [ ]:
def bfs_levels(graph, start):
    queue = deque([(start, 0)])
    visited = {start}
    levels = {}

    while queue:
        current, hop = queue.popleft()
        if hop not in levels:
            levels[hop] = []
        levels[hop].append(current)

        for next_device in graph[current]:
            if next_device not in visited:
                visited.add(next_device)
                queue.append((next_device, hop + 1))
    return levels

levels = bfs_levels(network, "N28")
print("Hop | Devices | Count")
for hop, devices in levels.items():
    print(hop, "|", ", ".join(devices), "|", len(devices))
print("Total:", sum(len(devices) for devices in levels.values()))

## Question 3 — Infection after 2, 3, and 4 hours
One hop takes one hour. Use the levels already found.

In [ ]:
def devices_within(graph, start, max_hours):
    levels = bfs_levels(graph, start)
    infected = []
    for hour in levels:
        if hour <= max_hours:
            infected.extend(levels[hour])
    return infected

for hours in [2, 3, 4]:
    infected = devices_within(network, "N28", hours)
    print("After", hours, "hours:", ", ".join(infected))
    print("Total:", len(infected))

## Question 4 — When does N08 get infected? Who is last?

In [ ]:
levels = bfs_levels(network, "N28")
for hour in levels:
    if "N08" in levels[hour]:
        print("Database N08 infected at hour:", hour)

last_hour = max(levels)
print("Last infection hour:", last_hour)
print("Last devices:", ", ".join(levels[last_hour]))

## Question 5 — Shortest BFS paths to N08 and N16
The `parent` dictionary remembers where each device was reached from.

In [ ]:
def bfs_shortest_path(graph, start, goal):
    queue = deque([start])
    parent = {start: None}

    while queue:
        current = queue.popleft()
        if current == goal:
            break
        for next_device in graph[current]:
            if next_device not in parent:
                parent[next_device] = current
                queue.append(next_device)

    if goal not in parent:
        return []
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = parent[current]
    path.reverse()
    return path

for goal in ["N08", "N16"]:
    path = bfs_shortest_path(network, "N28", goal)
    print("To", device_names[goal])
    print(" -> ".join(device_names[x] for x in path))
    print("Hops:", len(path) - 1)

## Question 6 — DFS path versus BFS path
DFS uses recursion and explores one route deeply before another.

In [ ]:
def dfs_path(graph, start, goal):
    visited = set()

    def search(current, path):
        visited.add(current)
        if current == goal:
            return path
        for next_device in graph[current]:
            if next_device not in visited:
                answer = search(next_device, path + [next_device])
                if answer:
                    return answer
        return []

    return search(start, [start])

bfs_route = bfs_shortest_path(network, "N28", "N08")
dfs_route = dfs_path(network, "N28", "N08")
for method, route in [("BFS", bfs_route), ("DFS", dfs_route)]:
    print(method, "path:", " -> ".join(device_names[x] for x in route))
    print("Hops:", len(route) - 1)
print("BFS guarantees the shortest path; DFS does not.")

## Question 7 — Farthest devices from Internet Gateway (N01)

In [ ]:
levels = bfs_levels(network, "N01")
last_hop = max(levels)
print("Maximum hops:", last_hop)
for device in levels[last_hop]:
    print(device, "-", device_names[device])

## Question 8 — Remove one device and check reachability
The removed device itself is **not** counted as unreachable.

In [ ]:
def bfs_without_node(graph, start, removed):
    if start == removed:
        return set()
    queue = deque([start])
    visited = {start}
    while queue:
        current = queue.popleft()
        for next_device in graph[current]:
            if next_device != removed and next_device not in visited:
                visited.add(next_device)
                queue.append(next_device)
    return visited

for removed in ["N04", "N03", "N02"]:
    reachable = bfs_without_node(network, "N01", removed)
    unreachable = sorted(set(network) - reachable - {removed})
    print("\nRemoved:", removed, device_names[removed])
    print("Unreachable count:", len(unreachable))
    print("Unreachable devices:", ", ".join(unreachable) or "None")
print("\nMost critical: N02 (Firewall-Main)")

## Question 9 — Isolate one device to slow infection
Isolate N27, the only direct neighbor of infected N28.

In [ ]:
isolated = "N27"
infected = bfs_without_node(network, "N28", isolated)
protected = set(network) - infected - {isolated}
print("Isolated:", isolated, device_names[isolated])
print("Infected:", ", ".join(sorted(infected)))
print("Infected count:", len(infected))
print("Protected count:", len(protected))
print("Reason: N27 connects infected N28 to the rest of the network.")